# Round 2 training on Colab (thin wrapper: every step is a script in `src/`)

1. Upload `colab_bundle.zip` (from `python src/make_colab_bundle.py`) to **MyDrive/indic_stance/**.
2. Runtime > Change runtime type > **T4 GPU**.
3. Runtime > **Run all**. If the session disconnects, run all again: finished runs are skipped.
4. Download **MyDrive/indic_stance/colab_results.zip** and unzip it in the repository root.

Optional: add a Colab secret `HF_TOKEN` (key icon, left bar) from a Hugging Face account that accepted the IndicTrans2 terms; without it translation uses NLLB-200 (recorded in every translated row).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU'
print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/indic_stance'

In [ ]:
# unpack the bundle; outputs live on Drive (symlinks) so a disconnect loses nothing
import os
!rm -rf /content/NLP && mkdir -p /content/NLP && unzip -q -o {DRIVE}/colab_bundle.zip -d /content/NLP
%cd /content/NLP
for d in ['outputs/preds', 'outputs/runs', 'data/translated', 'data/review', 'models']:
    os.makedirs(f'{DRIVE}/work/{d}', exist_ok=True)
    os.makedirs(os.path.dirname(d) or '.', exist_ok=True)
    os.symlink(f'{DRIVE}/work/{d}', d)
!ls -R data | head -30

In [ ]:
!pip install -q -r requirements-colab.txt
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    !pip install -q IndicTransToolkit || echo 'IndicTransToolkit failed: NLLB will be used'
except Exception:
    print('No HF_TOKEN secret: fine for IndicBERT v2, MuRIL and NLLB (IndicTrans2 is gated and needs it)')
!python src/train_transformer.py && python src/data.py | tail -1   # self-checks

In [ ]:
# translation + all 8 runs in priority order; finished runs are skipped; results zipped to Drive at the end
!python -u src/run_all.py --results-zip {DRIVE}/colab_results.zip

In [ ]:
!ls -la {DRIVE}/colab_results.zip && cat data/translated/translation_info.json